# 10. OPTIMIZE と Liquid Clustering - ファイルの置き方を整える

`01`〜`09` は、データを「入れる」話でした。
入れ続けると、テーブルは少しずつ読みにくい形になっていきます。

原因は **ファイルが増えること** です。
Deltaテーブルの実体はParquetファイルの集まりで、書き込むたびに新しいファイルが増えます。
ストリーミングやマイクロバッチのように少しずつ書くと、**小さいファイルが大量にできます。**

読むときは、その全部を開いて回ることになります。
数十行のために数十個のファイルを開くのは無駄です。1つにまとまっていれば1回で済みます。

このノートブックで確かめること:

1. 書き込みを繰り返すと、ファイルがどう増えるか
2. `OPTIMIZE` でまとめると何が変わるか
3. Z-ORDER と Liquid Clustering は何が違うか
4. 自分で実行するのか、任せるのか

**前提**: `00_setup` を実行済みであること。`04` を読んでいること。


## 準備


In [ ]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [ ]:
import random

CATALOG = "tech_survey"

# クラスタリングを指定しないテーブル
TABLE = f"{CATALOG}.silver.optimize_orders"

# Liquid Clustering を指定するテーブル。3章で使う
CLUSTERED = f"{CATALOG}.silver.optimize_clustered"

SCHEMA = "order_id STRING, product STRING, amount INT"
PRODUCTS = ["laptop", "monitor", "keyboard"]

## 1. 小さいファイルを作る

5行ずつ、10回に分けて書き込みます。
日次バッチを10日ぶん流した、あるいはストリーミングで10マイクロバッチ処理した、と思ってください。

合計50行です。ファイルが何個できるか予想してみてください。


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
spark.sql(f"CREATE TABLE {TABLE} ({SCHEMA})")

# 1回の書き込みごとに、新しいファイルができる
for i in range(10):
    rows = []
    for j in range(5):
        rows.append((f"{i}-{j}", random.choice(PRODUCTS), random.randint(1000, 50000)))

    (
        spark.createDataFrame(rows, SCHEMA)
        .write.format("delta")
        .mode("append")
        .saveAsTable(TABLE)
    )

print("書き込んだ行数:", spark.table(TABLE).count())

In [ ]:
# numFiles = 今そのテーブルが使っているファイルの数
display(spark.sql(f"DESCRIBE DETAIL {TABLE}").select("numFiles", "sizeInBytes"))

50行を保持するのにファイルが複数できています。

これが **スモールファイル問題** です。
ファイルごとにメタデータを読む固定コストがかかるので、
**行数が同じでもファイル数が多いほど遅くなります。**

ストリーミングでは特に起きやすくなります。
`02` で見たように、マイクロバッチごとに書き込みが走るからです。
1日1回のバッチなら年に365ファイルですが、5分ごとなら年に10万個を超えます。


## 2. `OPTIMIZE` でまとめる

`OPTIMIZE` は、小さいファイルを読んで **大きいファイルに書き直します。**
中身は変わりません。置き方だけが変わります。


In [ ]:
display(spark.sql(f"OPTIMIZE {TABLE}"))

In [ ]:
# まとめた後のファイル数を見る
display(spark.sql(f"DESCRIBE DETAIL {TABLE}").select("numFiles", "sizeInBytes"))

ファイルがまとまったはずです。行数は変わっていません。

注意してほしいのは、**古いファイルが消えたわけではない** ことです。
Deltaは「今はこの新しいファイルが有効」と記録し直しただけで、元のファイルはまだ置かれています。
だから `OPTIMIZE` の前の状態にも戻れます。この話は `11` で扱います。

履歴を見ると、`OPTIMIZE` が1つの操作として記録されています。


In [ ]:
display(
    spark.sql(f"DESCRIBE HISTORY {TABLE}").select("version", "operation", "operationMetrics").limit(5)
)

## 3. 並べ方を決める - Z-ORDER と Liquid Clustering

`OPTIMIZE` はファイルを **まとめる** だけでした。
もう一段あって、**どの行を同じファイルに入れるか** も決められます。

意味があるのは、絞り込んで読むときです。
`WHERE product = 'laptop'` で読むとき、laptopの行が1つのファイルに固まっていれば、
他のファイルは開かずに済みます。散らばっていると全部開くことになります。
Deltaはファイルごとに「この列の最小値と最大値」を持っていて、それで読み飛ばしを判断します。

指定の仕方が2つあります。

**Z-ORDER** は `OPTIMIZE` のときに指定します。

```sql
OPTIMIZE テーブル ZORDER BY (product)
```

実行したときだけ効きます。後から書き込んだぶんは並べ替えられないので、**定期的に打ち直す** ことになります。

**Liquid Clustering** はテーブルの性質として宣言します。`04` で使ったのがこれです。

```sql
CREATE TABLE テーブル (...) CLUSTER BY (product)
```

宣言しておけば、書き込みや `OPTIMIZE` のときに自動で寄せられます。


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {CLUSTERED}")

# CLUSTER BY を付けて作る。以降の書き込みはこのキーを意識して配置される
spark.sql(f"CREATE TABLE {CLUSTERED} ({SCHEMA}) CLUSTER BY (product)")

for i in range(10):
    rows = []
    for j in range(5):
        rows.append((f"{i}-{j}", random.choice(PRODUCTS), random.randint(1000, 50000)))

    (
        spark.createDataFrame(rows, SCHEMA)
        .write.format("delta")
        .mode("append")
        .saveAsTable(CLUSTERED)
    )

spark.sql(f"OPTIMIZE {CLUSTERED}")

# clusteringColumns に、宣言したキーが入っている
display(spark.sql(f"DESCRIBE DETAIL {CLUSTERED}").select("numFiles", "clusteringColumns"))

`clusteringColumns` に `product` が入っています。
テーブル自身が「このキーで寄せる」ことを覚えている、という点がZ-ORDERとの一番の違いです。

| | Z-ORDER | Liquid Clustering |
|---|---|---|
| どこに書くか | `OPTIMIZE` のたびに指定 | テーブルの定義 |
| 後からの書き込み | 次に `OPTIMIZE` するまで並ばない | 自動で寄せられる |
| キーの変更 | 次回から別のキーで打てる | `ALTER TABLE ... CLUSTER BY` で変更でき、既存データの書き直しが要らない |
| パーティションとの併用 | できる | できない (どちらか) |

現在のDatabricksは、新しいテーブルには **Liquid Clustering を推奨** しています。
Z-ORDERは、既存のテーブルで使われているのを読む機会のほうが多いはずです。

`04` で触れたパーティションも含めて整理すると、こうなります。

- **パーティション** … フォルダを物理的に分ける。後から変えられない。古い方式
- **Z-ORDER** … ファイルの中身の並びを整える。打ち直しが要る
- **Liquid Clustering** … 同じことをテーブルの性質として宣言する。後から変えられる


## 4. 自分でやるか、任せるか

ここまで `OPTIMIZE` を手で打ってきましたが、実運用で毎回打つのは現実的ではありません。
Databricksには **Predictive Optimization** という仕組みがあり、
アクセスの傾向を見て `OPTIMIZE` や `VACUUM` を自動で実行してくれます。

| | 手で打つ | 任せる |
|---|---|---|
| いつ | 大量に入れた直後など、タイミングが分かっているとき | 普段 |
| 手間 | ジョブに組み込む必要がある | 無し |
| コスト | 打った分だけ | 必要と判断されたときだけ |

**基本は任せる** で問題ありません。
手で打つ意味があるのは「これから大きな読み取りが走る」と分かっている場合です。
バックフィルで大量に書き込んだ直後に1回入れておく、といった使い方になります。

逆に **書き込みのたびに `OPTIMIZE` を打つのは無駄** です。
まとめる作業自体が読み書きを伴うので、細かく打つとコストだけが増えます。


## 考えてみる

- ファイルがまとまったのに、テーブルのサイズはほとんど減っていません。なぜでしょうか
- Liquid Clustering のキーには、どんな列を選ぶとよいでしょうか
- `OPTIMIZE` を打った直後に、同じテーブルへ5行だけ追記したらどうなりますか


### 答え

**Q1. サイズが減らない理由**

`OPTIMIZE` は **中身を捨てていない** からです。50行は50行のまま、入れ物を詰め替えただけです。

減るのは「ファイルの数」であって「データの量」ではありません。
効くのは、読むときの手間のほうです。

**Q2. キーの選び方**

**絞り込みに使う列** です。`WHERE` や `JOIN` の条件に出てくる列を選びます。

逆に選んではいけないのは、絞り込みに使わない列です。
寄せるコストだけかかって、読むときの得がありません。

値の種類が極端に少ない列 (例: 2種類しかないフラグ) も効きません。
半分のファイルしか飛ばせないので、労力に見合いません。

**Q3. 追記した直後**

**小さいファイルが1つ増えます。** まとめた大きいファイルと、新しい小さいファイルが並びます。

`OPTIMIZE` は打った時点のスナップショットに対する操作なので、後から来たぶんには効きません。
だから定期的に実行するか、Predictive Optimization に任せることになります。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。


In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {CLUSTERED}")